#### LSTM Experiment — RUL Prediction 

Benchmarking a 2-layer LSTM against the XGBoost baseline.
- Input: sliding window sequences (window=30 cycles) over sensor readings
- Target: RUL at the last timestep of each window
- Evaluation: RMSE, MAE, R² on the same test set used for XGBoost

In [2]:
import numpy as np
import pandas as pd
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

np.random.seed(42)
tf.random.set_seed(42)

##### 1. Load Engineered Features

In [3]:
DATA_DIR   = os.path.join("..", "data")
MODELS_DIR = os.path.join("..", "models")
os.makedirs(MODELS_DIR, exist_ok=True)

train = pd.read_csv(os.path.join(DATA_DIR, "train_features.csv"))
test  = pd.read_csv(os.path.join(DATA_DIR, "test_features.csv"))

NON_FEATURE = {"unit", "cycle", "RUL"}
feature_cols = [c for c in train.columns if c not in NON_FEATURE]

print(f"Train shape : {train.shape}")
print(f"Test  shape : {test.shape}")
print(f"Features    : {len(feature_cols)}")

Train shape : (20631, 48)
Test  shape : (13096, 48)
Features    : 45


##### 2. Build Sliding Window Sequences
Each sample is a (window_size, n_features) tensor.  
The label is the RUL at the **last** timestep of the window.

In [4]:
WINDOW = 30

def build_sequences(df, feature_cols, window=WINDOW):
    X_list, y_list = [], []
    for unit_id, group in df.groupby("unit"):
        group = group.sort_values("cycle")
        X_unit = group[feature_cols].values
        y_unit = group["RUL"].values
        for i in range(len(group) - window + 1):
            X_list.append(X_unit[i : i + window])
            y_list.append(y_unit[i + window - 1])
    return np.array(X_list), np.array(y_list)


scaler = StandardScaler()
train_scaled = train.copy()
test_scaled  = test.copy()
train_scaled[feature_cols] = scaler.fit_transform(train[feature_cols])
test_scaled[feature_cols]  = scaler.transform(test[feature_cols])

X_train, y_train = build_sequences(train_scaled, feature_cols)

test_last = test_scaled.loc[
    test_scaled.groupby("unit")["cycle"].apply(
        lambda x: x.index[x == x.max()][0]
    )
].reset_index(drop=True)

def build_test_sequences(df_full, test_last_df, feature_cols, window=WINDOW):
    X_list, y_list = [], []
    for _, row in test_last_df.iterrows():
        unit_id   = row["unit"]
        end_cycle = row["cycle"]
        group = df_full[df_full["unit"] == unit_id].sort_values("cycle")
        group = group[group["cycle"] <= end_cycle]
        X_unit = group[feature_cols].values
        y_unit = group["RUL"].values
        if len(group) >= window:
            X_list.append(X_unit[-window:])
            y_list.append(y_unit[-1])
        else:
            pad = np.zeros((window - len(group), len(feature_cols)))
            X_list.append(np.vstack([pad, X_unit]))
            y_list.append(y_unit[-1])
    return np.array(X_list), np.array(y_list)

X_test, y_test = build_test_sequences(test_scaled, test_last, feature_cols)

print(f"X_train : {X_train.shape}  |  y_train : {y_train.shape}")
print(f"X_test  : {X_test.shape}   |  y_test  : {y_test.shape}")

X_train : (17731, 30, 45)  |  y_train : (17731,)
X_test  : (100, 30, 45)   |  y_test  : (100,)


##### 3. Build & Train the LSTM

In [5]:
def build_lstm(input_shape):
    model = Sequential([
        LSTM(128, input_shape=input_shape, return_sequences=True),
        Dropout(0.2),
        LSTM(64, return_sequences=False),
        Dropout(0.2),
        Dense(32, activation="relu"),
        Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  loss="mse",
                  metrics=["mae"])
    return model

model = build_lstm((WINDOW, len(feature_cols)))
model.summary()

c:\Users\Asus\anaconda3\envs\pw\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 128)        │        89,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 140,609 (549.25 KB)

 Trainable params: 140,609 (549.25 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
early_stop = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
reduce_lr  = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-5)

history = model.fit(
    X_train, y_train,
    epochs          = 100,
    batch_size      = 256,
    validation_split= 0.1,
    callbacks       = [early_stop, reduce_lr],
    verbose         = 1
)

Epoch 1/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 8s 103ms/step - loss: 5663.5449 - mae: 64.4687 - val_loss: 3980.0608 - val_mae: 53.6997 - learning_rate: 0.0010
Epoch 2/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 5s 72ms/step - loss: 2217.3213 - mae: 38.0585 - val_loss: 1361.7046 - val_mae: 31.1151 - learning_rate: 0.0010
Epoch 3/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 5s 74ms/step - loss: 819.6230 - mae: 23.3950 - val_loss: 538.4000 - val_mae: 20.1768 - learning_rate: 0.0010
Epoch 4/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 7s 116ms/step - loss: 395.4547 - mae: 16.2028 - val_loss: 308.2628 - val_mae: 14.8042 - learning_rate: 0.0010
Epoch 5/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 5s 73ms/step - loss: 260.7324 - mae: 12.6435 - val_loss: 267.1152 - val_mae: 12.6591 - learning_rate: 0.0010
Epoch 6/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 6s 95ms/step - loss: 203.2094 - mae: 10.8569 - val_loss: 274.6770 - val_mae: 12.5682 - learning_rate: 0.0010
Epoch 7/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 4s 66ms/step - loss: 162.1773 - mae: 9.6848 - val_loss: 273.6220 - 

##### 4. Evaluate LSTM on Test Set

In [7]:
y_pred_lstm = model.predict(X_test).flatten()
y_pred_lstm = np.clip(y_pred_lstm, 0, None)

rmse_lstm = np.sqrt(mean_squared_error(y_test, y_pred_lstm))
mae_lstm  = mean_absolute_error(y_test, y_pred_lstm)
r2_lstm   = r2_score(y_test, y_pred_lstm)

XGB_RMSE = 19.25
XGB_MAE  = 14.30
XGB_R2   = 0.785

print("=" * 45)
print(f"  {'Metric':<10}  {'XGBoost':>10}  {'LSTM':>10}")
print("=" * 45)
print(f"  {'RMSE':<10}  {XGB_RMSE:>10.4f}  {rmse_lstm:>10.4f}")
print(f"  {'MAE':<10}  {XGB_MAE:>10.4f}  {mae_lstm:>10.4f}")
print(f"  {'R²':<10}  {XGB_R2:>10.4f}  {r2_lstm:>10.4f}")
print("=" * 45)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
  Metric         XGBoost        LSTM
  RMSE           19.2500     16.7058
  MAE            14.3000     12.8750
  R²              0.7850      0.8384


##### 5. Comparison Visualization

In [8]:
BG_DARK  = "#1a1a2e"
BG_CARD  = "#16213e"
ORANGE   = "#f97316"
BLUE     = "#38bdf8"
TEXT     = "#e2e8f0"
SUBTEXT  = "#94a3b8"

metrics      = ["RMSE", "MAE", "R²"]
xgb_values   = [XGB_RMSE, XGB_MAE, XGB_R2]
lstm_values  = [rmse_lstm, mae_lstm, r2_lstm]

fig = plt.figure(figsize=(18, 12), facecolor=BG_DARK)
fig.suptitle("XGBoost vs LSTM — RUL Prediction on CMAPSS FD001",
             color=ORANGE, fontsize=16, fontweight="bold", y=0.98)

gs = fig.add_gridspec(2, 3, hspace=0.42, wspace=0.35,
                      top=0.92, bottom=0.08, left=0.07, right=0.97)

# --- Row 1: metric bar charts ---
for i, (metric, xgb_val, lstm_val) in enumerate(zip(metrics, xgb_values, lstm_values)):
    ax = fig.add_subplot(gs[0, i])
    ax.set_facecolor(BG_CARD)

    bars = ax.bar(["XGBoost", "LSTM"], [xgb_val, lstm_val],
                  color=[ORANGE, BLUE], width=0.45, edgecolor="none")

    for bar, val in zip(bars, [xgb_val, lstm_val]):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(xgb_val, lstm_val) * 0.02,
                f"{val:.3f}", ha="center", va="bottom",
                color=TEXT, fontsize=11, fontweight="bold")

    ax.set_title(metric, color=ORANGE, fontsize=13, pad=10)
    ax.set_facecolor(BG_CARD)
    ax.tick_params(colors=TEXT, labelsize=10)
    ax.spines[:].set_color("#334155")
    ax.set_ylim(0, max(xgb_val, lstm_val) * 1.25)
    for spine in ax.spines.values():
        spine.set_linewidth(0.6)
    ax.yaxis.label.set_color(TEXT)

# --- Row 2 left+mid: Predicted vs Actual overlaid ---
ax2 = fig.add_subplot(gs[1, :2])
ax2.set_facecolor(BG_CARD)

sort_idx   = np.argsort(y_test)
y_sorted   = y_test[sort_idx]
yp_xgb_s   = np.array([XGB_RMSE] * len(y_test))  # placeholder line
yp_lstm_s  = y_pred_lstm[sort_idx]

ax2.plot(y_sorted, color=SUBTEXT, linewidth=1.5, label="Actual RUL", linestyle="--")
ax2.plot(yp_lstm_s, color=BLUE,   linewidth=1.2, label=f"LSTM  (RMSE={rmse_lstm:.2f})", alpha=0.85)

ax2.set_title("Predicted vs Actual RUL — LSTM", color=ORANGE, fontsize=12, pad=10)
ax2.set_xlabel("Engine Index (sorted by Actual RUL)", color=TEXT, fontsize=10)
ax2.set_ylabel("RUL (cycles)", color=TEXT, fontsize=10)
ax2.tick_params(colors=TEXT, labelsize=9)
ax2.spines[:].set_color("#334155")
ax2.legend(facecolor=BG_CARD, labelcolor=TEXT, fontsize=9)

# --- Row 2 right: Training loss curve ---
ax3 = fig.add_subplot(gs[1, 2])
ax3.set_facecolor(BG_CARD)

ax3.plot(history.history["loss"],     color=ORANGE, linewidth=1.4, label="Train Loss")
ax3.plot(history.history["val_loss"], color=BLUE,   linewidth=1.4, label="Val Loss",  linestyle="--")
ax3.set_title("LSTM Training Loss", color=ORANGE, fontsize=12, pad=10)
ax3.set_xlabel("Epoch", color=TEXT, fontsize=10)
ax3.set_ylabel("MSE Loss", color=TEXT, fontsize=10)
ax3.tick_params(colors=TEXT, labelsize=9)
ax3.spines[:].set_color("#334155")
ax3.legend(facecolor=BG_CARD, labelcolor=TEXT, fontsize=9)

out_path = os.path.join(MODELS_DIR, "model_comparison.png")
plt.savefig(out_path, dpi=150, bbox_inches="tight", facecolor=BG_DARK)
plt.show()
print(f"Saved: {out_path}")

Saved: ..\models\model_comparison.png


C:\Users\Asus\AppData\Local\Temp\ipykernel_39760\2921839619.py:76: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


##### 6. Conclusion

| Metric | XGBoost | LSTM |
|--------|---------|------|
| RMSE   | 19.25   |  16.71|
| MAE    | 14.30   | 12.88 |
| R²     | 0.785   | 0.838  |

